# IPL SQL Setup — Step by Step for Power BI

This notebook creates **exactly six final tables** from the five EDA-reviewed CSVs:

- **Dimensions:** `dim_teams`, `dim_players`, `dim_venues`, `dim_date`
- **Facts:** `fact_matches`, `fact_deliveries`

Each step solves one problem in a short code cell. The team-alias CSV is used only as a reference during review; it is **not exported as a model table**. The notebook exports only the six tables to `DATA/POWERBI_TABLES/`.

## Step 1 — Import the libraries

Keep the setup minimal. Pandas handles the tables; NumPy is used for simple flags.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

## Step 2 — Set the project and export folders

Run the notebook from your project root where possible. The output folder is created automatically.

In [2]:
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "DATA" / "POWERBI_TABLES"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Power BI export folder:", OUTPUT_DIR)

Project: /mnt/data
Power BI export folder: /mnt/data/DATA/POWERBI_TABLES


## Step 3 — Find the five EDA CSV files

This checks the common cleaned-data folders and the filenames produced in the EDA steps. No source file is overwritten.

In [3]:
SEARCH_DIRS = [
    PROJECT_ROOT / "data" / "cleaned",
    PROJECT_ROOT,
    PROJECT_ROOT.parent / "data" / "cleaned",
    PROJECT_ROOT.parent,
]

SOURCE_NAMES = {
    "deliveries": ["ball_by_ball_data_cleaned.csv", "ball_by_ball_data_clean.csv", "ball_by_ball_data(1).csv"],
    "matches": ["ipl_matches_data_cleaned.csv", "ipl_matches_data_clean.csv", "ipl_matches_data(1).csv"],
    "players": ["players_data_cleaned.csv", "players-data-updated_clean.csv", "players-data-updated(1).csv"],
    "aliases": ["team_aliases_cleaned.csv", "team_aliases_clean.csv", "team_aliases(1).csv"],
    "teams": ["teams_data_cleaned.csv", "teams_data_clean.csv", "teams_data(1).csv"],
}

def find_source(names):
    for folder in SEARCH_DIRS:
        for name in names:
            path = folder / name
            if path.exists():
                return path
    raise FileNotFoundError(f"Could not find any of these files: {names}")

SOURCE_PATHS = {key: find_source(names) for key, names in SOURCE_NAMES.items()}
pd.DataFrame({"source": SOURCE_PATHS.keys(), "path": [str(p) for p in SOURCE_PATHS.values()]})

,source,path
0,deliveries,/mnt/data/ball_by_ball_data_cleaned.csv
1,matches,/mnt/data/ipl_matches_data_cleaned.csv
2,players,/mnt/data/players_data_cleaned.csv
3,aliases,/mnt/data/team_aliases_cleaned.csv
4,teams,/mnt/data/teams_data_cleaned.csv


## Step 4 — Load the source data

The alias table is loaded for reference only. The final model will not contain an alias dimension.

In [4]:
deliveries = pd.read_csv(SOURCE_PATHS["deliveries"], low_memory=False)
matches = pd.read_csv(SOURCE_PATHS["matches"])
players = pd.read_csv(SOURCE_PATHS["players"])
team_aliases = pd.read_csv(SOURCE_PATHS["aliases"])
teams = pd.read_csv(SOURCE_PATHS["teams"])

pd.DataFrame([
    ["deliveries", len(deliveries), len(deliveries.columns)],
    ["matches", len(matches), len(matches.columns)],
    ["players", len(players), len(players.columns)],
    ["team aliases (reference only)", len(team_aliases), len(team_aliases.columns)],
    ["teams", len(teams), len(teams.columns)],
], columns=["source", "rows", "columns"])

,source,rows,columns
0,deliveries,288226,30
1,matches,1212,23
2,players,799,6
3,team aliases (reference only),46,3
4,teams,16,2


## Step 5 — Standardize blank text

Whitespace-only strings become missing values. This is important for `bowl_style` and `field_pos`; no rows are deleted.

In [5]:
def clean_text(df):
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip().replace("", pd.NA)
    return df

deliveries = clean_text(deliveries)
matches = clean_text(matches)
players = clean_text(players)
team_aliases = clean_text(team_aliases)
teams = clean_text(teams)

## Step 6 — Check source keys and cricket arithmetic

Stop here if a source key or a tested score calculation is inconsistent.

In [6]:
assert matches["match_id"].is_unique
assert players["player_id"].is_unique
assert teams["team_id"].is_unique
assert team_aliases["alias_id"].is_unique

assert set(deliveries["match_id"]) == set(matches["match_id"])
assert (deliveries["total_runs"] == deliveries["batter_runs"] + deliveries["extras"]).all()

extra_cols = ["wide_ball_runs", "no_ball_runs", "leg_bye_runs", "bye_runs", "penalty_runs"]
assert (deliveries["extras"] == deliveries[extra_cols].sum(axis=1)).all()

print("Source key and run-arithmetic checks passed.")

Source key and run-arithmetic checks passed.


## Step 7 — Define the historical team-ID rules

The source reuses numeric IDs for historical team entries. Apply the same rules to match rows and delivery rows so they agree.

In [7]:
def standardize_team_ids(team_ids, seasons):
    team_ids = pd.to_numeric(team_ids).copy()
    seasons = pd.to_numeric(seasons)

    # Team ID 2 represents Deccan Chargers up to 2012.
    team_ids = team_ids.mask(team_ids.eq(2) & seasons.between(2008, 2012), 1068)

    # Team ID 4 represents the plural Rising Pune entry in 2016.
    team_ids = team_ids.mask(team_ids.eq(4) & seasons.eq(2016), 3604)
    return team_ids.astype("int64")

print("Deccan Chargers alias in source reference:")
display(team_aliases.loc[team_aliases["alias_name"].eq("Deccan Chargers")])

Deccan Chargers alias in source reference:


,alias_id,team_id,alias_name
17,18,2,Deccan Chargers


## Step 8 — Build `dim_teams`

Keep the existing IDs and canonical names. Do not renumber IDs simply because they are not sequential.

In [8]:
dim_teams = teams[["team_id", "team_name"]].copy()
dim_teams["team_id"] = dim_teams["team_id"].astype("int64")
dim_teams = dim_teams.sort_values("team_id").reset_index(drop=True)

display(dim_teams)

,team_id,team_name
0,1,Royal Challengers Bangalore
1,2,Sunrisers Hyderabad
2,3,Mumbai Indians
3,4,Rising Pune Supergiant
4,5,Gujarat Lions
5,6,Kolkata Knight Riders
6,129,Chennai Super Kings
7,134,Rajasthan Royals
8,252,Delhi Capitals
9,494,Punjab Kings


## Step 9 — Build `dim_players`

Keep missing bowling styles and field positions as unknown. Use `player_id` rather than names as the joining key.

In [9]:
dim_players = players[[
    "player_id", "player_name", "player_full_name", "bat_style", "bowl_style", "field_pos"
]].copy()

dim_players["player_id"] = dim_players["player_id"].astype("int64")
dim_players = dim_players.sort_values("player_id").reset_index(drop=True)
player_name_to_id = dict(zip(dim_players["player_name"], dim_players["player_id"]))

display(dim_players.head())

,player_id,player_name,player_full_name,bat_style,bowl_style,field_pos
0,1,CH Gayle,Christopher Henry Gayle,Left hand Bat,Right arm Offbreak,NaN
1,2,Mandeep Singh,Mandeep Singh,Right hand Bat,Right arm Medium,NaN
2,3,TM Head,Travis Michael Head,Left hand Bat,Right arm Offbreak,Wicketkeeper
3,4,KM Jadhav,Kedar Mahadav Jadhav,Right hand Bat,Right arm Offbreak,NaN
4,5,SR Watson,Shane Robert Watson,Right hand Bat,Right arm Fast medium,NaN


## Step 10 — Build `dim_venues`

The same venue label can appear with different city labels. Preserve each observed venue-and-city pair rather than merging them by guess.

In [10]:
venue_pairs = matches[["venue", "city"]].drop_duplicates()
venue_pairs = venue_pairs.sort_values(["venue", "city"]).reset_index(drop=True)
venue_pairs.insert(0, "venue_id", range(1, len(venue_pairs) + 1))

dim_venues = venue_pairs.rename(columns={"venue": "venue_name"})
venue_id_map = {
    (row.venue, row.city): row.venue_id
    for row in venue_pairs.itertuples(index=False)
}

display(dim_venues.head())
print("Venue-and-city combinations:", len(dim_venues))

,venue_id,venue_name,city
0,1,Arun Jaitley Stadium,Delhi
1,2,"Arun Jaitley Stadium, Delhi",Delhi
2,3,Barabati Stadium,Cuttack
3,4,"Barsapara Cricket Stadium, Guwahati",Guwahati
4,5,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow


Venue-and-city combinations: 61


## Step 11 — Build `dim_date`

Create one row for every calendar date between the first and last match. This supports Power BI date filtering and time intelligence.

In [11]:
matches["match_date"] = pd.to_datetime(matches["match_date"], errors="raise")
calendar = pd.date_range(matches["match_date"].min(), matches["match_date"].max(), freq="D")

dim_date = pd.DataFrame({"full_date": calendar})
dim_date["date_id"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype("int32")
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["month_number"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.strftime("%B")
dim_date["day_number"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.strftime("%A")
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["year_month"] = dim_date["full_date"].dt.strftime("%Y-%m")

display(dim_date.head())
print("Calendar days:", len(dim_date))

,full_date,date_id,year,month_number,month_name,day_number,day_name,quarter,year_month
0,2008-04-18,20080418,2008,4,April,18,Friday,2,2008-04
1,2008-04-19,20080419,2008,4,April,19,Saturday,2,2008-04
2,2008-04-20,20080420,2008,4,April,20,Sunday,2,2008-04
3,2008-04-21,20080421,2008,4,April,21,Monday,2,2008-04
4,2008-04-22,20080422,2008,4,April,22,Tuesday,2,2008-04


Calendar days: 6588


## Step 12 — Add date and venue keys to the match source

These keys let both fact tables connect directly to the four dimensions. There is no separate `dim_matches` table.

In [12]:
matches["date_id"] = matches["match_date"].dt.strftime("%Y%m%d").astype("int32")
matches["venue_id"] = [venue_id_map[(v, c)] for v, c in zip(matches["venue"], matches["city"])]

print("Date keys added:", matches["date_id"].notna().all())
print("Venue keys added:", matches["venue_id"].notna().all())

Date keys added: True
Venue keys added: True


## Step 13 — Start `fact_matches`

Keep one row per match. Rename team-role columns so their foreign-key role is obvious.

In [13]:
fact_matches = matches[[
    "match_id", "season_id", "date_id", "venue_id", "match_number",
    "team1", "team2", "toss_winner", "toss_decision", "match_winner",
    "win_by_runs", "win_by_wickets", "result", "player_of_match"
]].copy()

fact_matches = fact_matches.rename(columns={
    "team1": "team1_id",
    "team2": "team2_id",
    "toss_winner": "toss_winner_id",
    "match_winner": "match_winner_id",
    "player_of_match": "player_of_match_id",
})

fact_matches["match_id"] = fact_matches["match_id"].astype("int64")

## Step 14 — Standardize team IDs in `fact_matches`

Apply the historical rules to all four team roles. The source data is left unchanged.

In [14]:
for col in ["team1_id", "team2_id", "toss_winner_id", "match_winner_id"]:
    fact_matches[col] = standardize_team_ids(fact_matches[col], fact_matches["season_id"])

print("Historical team IDs standardized in fact_matches.")

Historical team IDs standardized in fact_matches.


## Step 15 — Clean match result fields

Null margins on ties/no-results are retained. The extra fields below are derived only from values already in the source.

In [15]:
fact_matches["match_number"] = pd.to_numeric(fact_matches["match_number"], errors="coerce").astype("Int64")
fact_matches["player_of_match_id"] = pd.to_numeric(fact_matches["player_of_match_id"], errors="coerce").astype("Int64")
fact_matches["win_by_runs"] = pd.to_numeric(fact_matches["win_by_runs"], errors="coerce").astype("Int64")
fact_matches["win_by_wickets"] = pd.to_numeric(fact_matches["win_by_wickets"], errors="coerce").astype("Int64")

fact_matches["win_type"] = "Unknown"
fact_matches.loc[fact_matches["result"].eq("tie"), "win_type"] = "Tie"
fact_matches.loc[fact_matches["result"].eq("no result"), "win_type"] = "No Result"
fact_matches.loc[fact_matches["win_by_runs"].gt(0), "win_type"] = "Batting First"
fact_matches.loc[fact_matches["win_by_wickets"].gt(0), "win_type"] = "Chasing"

## Step 16 — Calculate whether the toss winner won

This is unknown for ties/no-results rather than forcing those cases to 0.

In [16]:
fact_matches["toss_winner_won_match"] = pd.Series(pd.NA, index=fact_matches.index, dtype="Int64")
win_rows = fact_matches["result"].eq("win")
fact_matches.loc[win_rows, "toss_winner_won_match"] = (
    fact_matches.loc[win_rows, "toss_winner_id"].eq(fact_matches.loc[win_rows, "match_winner_id"]).astype("int8")
)

fact_matches = fact_matches.sort_values("match_id").reset_index(drop=True)
display(fact_matches.head())

,match_id,season_id,date_id,venue_id,match_number,team1_id,team2_id,toss_winner_id,toss_decision,match_winner_id,win_by_runs,win_by_wickets,result,player_of_match_id,win_type,toss_winner_won_match
0,335982,2008,20080418,25,1,1,6,1,field,6,140,<NA>,win,46,Batting First,0
1,335983,2008,20080419,44,2,494,129,129,bat,129,33,<NA>,win,11943,Batting First,1
2,335984,2008,20080419,18,3,252,134,134,bat,252,<NA>,9,win,11982,Chasing,0
3,335985,2008,20080420,59,5,3,1,3,bat,1,<NA>,5,win,11935,Chasing,0
4,335986,2008,20080420,16,4,6,1068,1068,bat,6,<NA>,5,win,11922,Chasing,0


## Step 17 — Attach match context to the delivery rows

Each delivery inherits the date key and venue key of its match. This avoids a seventh table.

In [17]:
match_context = fact_matches[["match_id", "season_id", "date_id", "venue_id"]]

fact_deliveries = deliveries.merge(
    match_context,
    on="match_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_match")
)

assert fact_deliveries["date_id"].notna().all()
assert fact_deliveries["venue_id"].notna().all()
assert fact_deliveries["season_id"].eq(fact_deliveries["season_id_match"]).all() if "season_id_match" in fact_deliveries else True

fact_deliveries = fact_deliveries.drop(columns=["season_id_match"], errors="ignore")
print("Match date and venue context attached to every delivery.")

Match date and venue context attached to every delivery.


## Step 18 — Map player names to player IDs

All non-null batter, bowler and non-striker names should map to `dim_players`. Player-out is naturally blank when no wicket falls.

In [18]:
fact_deliveries["batter_id"] = fact_deliveries["batter"].map(player_name_to_id).astype("Int64")
fact_deliveries["bowler_id"] = fact_deliveries["bowler"].map(player_name_to_id).astype("Int64")
fact_deliveries["non_striker_id"] = fact_deliveries["non_striker"].map(player_name_to_id).astype("Int64")
fact_deliveries["player_out_id"] = fact_deliveries["player_out"].map(player_name_to_id).astype("Int64")

assert fact_deliveries[["batter_id", "bowler_id", "non_striker_id"]].notna().all().all()
assert fact_deliveries.loc[fact_deliveries["player_out"].notna(), "player_out_id"].notna().all()
print("Player-name mapping passed.")

Player-name mapping passed.


## Step 19 — Standardize team IDs in `fact_deliveries`

Use the match season when applying the historical mapping rules.

In [19]:
fact_deliveries["batting_team_id"] = standardize_team_ids(
    fact_deliveries["team_batting"], fact_deliveries["season_id"]
)
fact_deliveries["bowling_team_id"] = standardize_team_ids(
    fact_deliveries["team_bowling"], fact_deliveries["season_id"]
)

assert fact_deliveries["batting_team_id"].ne(fact_deliveries["bowling_team_id"]).all()
print("Delivery team IDs standardized.")

Delivery team IDs standardized.


## Step 20 — Create delivery flags for analysis

These are small, transparent derivations. A dot ball uses `total_runs == 0`, not `batter_runs == 0`.

In [20]:
fact_deliveries["is_legal_ball"] = ~(
    fact_deliveries["is_wide_ball"] | fact_deliveries["is_no_ball"]
)
fact_deliveries["is_dot_ball"] = fact_deliveries["total_runs"].eq(0)
fact_deliveries["is_boundary_4"] = fact_deliveries["batter_runs"].eq(4)
fact_deliveries["is_boundary_6"] = fact_deliveries["batter_runs"].eq(6)

fact_deliveries["match_phase"] = np.select(
    [fact_deliveries["is_super_over"], fact_deliveries["over_number"].le(5), fact_deliveries["over_number"].le(14)],
    ["Super Over", "Powerplay", "Middle"],
    default="Death"
)

display(fact_deliveries[["over_number", "total_runs", "is_legal_ball", "is_dot_ball", "match_phase"]].head())

,over_number,total_runs,is_legal_ball,is_dot_ball,match_phase
0,0,1,True,False,Powerplay
1,0,0,True,True,Powerplay
2,0,1,False,False,Powerplay
3,0,0,True,True,Powerplay
4,0,0,True,True,Powerplay


## Step 21 — Create a unique delivery key

The source has a natural delivery coordinate. Sort first, then add a simple surrogate `delivery_id` for database use.

In [21]:
fact_deliveries = fact_deliveries.sort_values(
    ["match_id", "innings", "over_number", "ball_number"], kind="mergesort"
).reset_index(drop=True)

legal_ball_count = fact_deliveries.groupby(["match_id", "innings"])["is_legal_ball"].cumsum()
fact_deliveries["legal_ball_number"] = legal_ball_count.where(
    fact_deliveries["is_legal_ball"]
).astype("Int64")
fact_deliveries.insert(0, "delivery_id", range(1, len(fact_deliveries) + 1))

fact_deliveries["delivery_id"] = fact_deliveries["delivery_id"].astype("int64")
print("Unique delivery IDs:", fact_deliveries["delivery_id"].is_unique)

Unique delivery IDs: True


## Step 22 — Select the final delivery fact columns

Keep source measures and relevant foreign keys. Names stay in `dim_players` and team labels stay in `dim_teams`.

In [22]:
delivery_columns = [
    "delivery_id", "match_id", "season_id", "date_id", "venue_id",
    "innings", "over_number", "ball_number", "legal_ball_number",
    "batter_id", "bowler_id", "non_striker_id", "player_out_id",
    "batting_team_id", "bowling_team_id",
    "batter_runs", "extras", "total_runs", "is_wicket", "wicket_kind", "fielders_involved",
    "is_wide_ball", "is_no_ball", "is_leg_bye", "is_bye", "is_penalty",
    "wide_ball_runs", "no_ball_runs", "leg_bye_runs", "bye_runs", "penalty_runs",
    "is_super_over", "is_legal_ball", "is_dot_ball", "is_boundary_4", "is_boundary_6",
    "match_phase", "batsman_type", "bowler_type"
]
fact_deliveries = fact_deliveries[delivery_columns].copy()

print("fact_deliveries:", fact_deliveries.shape)
display(fact_deliveries.head())

fact_deliveries: (288226, 39)


,delivery_id,match_id,season_id,date_id,venue_id,innings,over_number,ball_number,legal_ball_number,batter_id,bowler_id,non_striker_id,player_out_id,batting_team_id,bowling_team_id,batter_runs,extras,total_runs,is_wicket,wicket_kind,fielders_involved,is_wide_ball,is_no_ball,is_leg_bye,is_bye,is_penalty,wide_ball_runs,no_ball_runs,leg_bye_runs,bye_runs,penalty_runs,is_super_over,is_legal_ball,is_dot_ball,is_boundary_4,is_boundary_6,match_phase,batsman_type,bowler_type
0,1,335982,2008,20080418,25,1,0,0,1,11919,52,46,<NA>,6,1,0,1,1,False,NaN,NaN,False,False,True,False,False,0,0,1,0,0,False,True,False,False,False,Powerplay,Left hand Bat,Right arm Medium
1,2,335982,2008,20080418,25,1,0,1,2,46,52,11919,<NA>,6,1,0,0,0,False,NaN,NaN,False,False,False,False,False,0,0,0,0,0,False,True,True,False,False,Powerplay,Right hand Bat,Right arm Medium
2,3,335982,2008,20080418,25,1,0,2,<NA>,46,52,11919,<NA>,6,1,0,1,1,False,NaN,NaN,True,False,False,False,False,1,0,0,0,0,False,False,False,False,False,Powerplay,Right hand Bat,Right arm Medium
3,4,335982,2008,20080418,25,1,0,3,3,46,52,11919,<NA>,6,1,0,0,0,False,NaN,NaN,False,False,False,False,False,0,0,0,0,0,False,True,True,False,False,Powerplay,Right hand Bat,Right arm Medium
4,5,335982,2008,20080418,25,1,0,4,4,46,52,11919,<NA>,6,1,0,0,0,False,NaN,NaN,False,False,False,False,False,0,0,0,0,0,False,True,True,False,False,Powerplay,Right hand Bat,Right arm Medium


## Step 23 — Check that there are exactly six final tables

No `dim_matches`, `dim_team_aliases`, QA table, or manifest is created as a final model table.

In [23]:
model_tables = {
    "dim_teams": dim_teams,
    "dim_players": dim_players,
    "dim_venues": dim_venues,
    "dim_date": dim_date,
    "fact_matches": fact_matches,
    "fact_deliveries": fact_deliveries,
}

assert len(model_tables) == 6
for name, table in model_tables.items():
    print(f"{name:18s} {len(table):>8,} rows × {len(table.columns):>2} columns")

dim_teams                16 rows ×  2 columns
dim_players             799 rows ×  6 columns
dim_venues               61 rows ×  3 columns
dim_date              6,588 rows ×  9 columns
fact_matches          1,212 rows × 16 columns
fact_deliveries     288,226 rows × 39 columns


## Step 24 — Validate keys and relationships

These checks confirm that the fact tables only refer to keys that exist in the dimensions.

In [24]:
team_ids = set(dim_teams["team_id"])
player_ids = set(dim_players["player_id"])
venue_ids = set(dim_venues["venue_id"])
date_ids = set(dim_date["date_id"])

assert dim_teams["team_id"].is_unique
assert dim_players["player_id"].is_unique
assert dim_venues["venue_id"].is_unique
assert dim_date["date_id"].is_unique
assert fact_matches["match_id"].is_unique
assert fact_deliveries["delivery_id"].is_unique

assert fact_matches["date_id"].isin(date_ids).all()
assert fact_deliveries["date_id"].isin(date_ids).all()
assert fact_matches["venue_id"].isin(venue_ids).all()
assert fact_deliveries["venue_id"].isin(venue_ids).all()

print("Dimension keys and date/venue relationships passed.")

Dimension keys and date/venue relationships passed.


## Step 25 — Validate team and player foreign keys

Missing `player_out_id` and `player_of_match_id` are allowed where the source has no dismissal/award value.

In [25]:
match_team_cols = ["team1_id", "team2_id", "toss_winner_id", "match_winner_id"]
delivery_team_cols = ["batting_team_id", "bowling_team_id"]

assert fact_matches[match_team_cols].apply(lambda col: col.isin(team_ids)).all().all()
assert fact_deliveries[delivery_team_cols].apply(lambda col: col.isin(team_ids)).all().all()
assert fact_matches["player_of_match_id"].dropna().isin(player_ids).all()

for col in ["batter_id", "bowler_id", "non_striker_id"]:
    assert fact_deliveries[col].dropna().isin(player_ids).all()
assert fact_deliveries["player_out_id"].dropna().isin(player_ids).all()

print("Team and player foreign-key checks passed.")

Team and player foreign-key checks passed.


## Step 26 — Validate match and delivery logic

Re-check key cricket rules after all mappings have been applied.

In [26]:
assert (fact_matches["team1_id"] != fact_matches["team2_id"]).all()
assert fact_matches["toss_winner_id"].eq(fact_matches["team1_id"]) .add(
    fact_matches["toss_winner_id"].eq(fact_matches["team2_id"])
).all()
assert fact_matches["match_winner_id"].eq(fact_matches["team1_id"]).add(
    fact_matches["match_winner_id"].eq(fact_matches["team2_id"])
).all()

assert not fact_deliveries.duplicated(["match_id", "innings", "over_number", "ball_number"]).any()
assert fact_deliveries["total_runs"].eq(fact_deliveries["batter_runs"] + fact_deliveries["extras"]).all()
assert fact_deliveries["extras"].eq(fact_deliveries[["wide_ball_runs", "no_ball_runs", "leg_bye_runs", "bye_runs", "penalty_runs"]].sum(axis=1)).all()

print("Match, delivery-grain and cricket-logic checks passed.")

Match, delivery-grain and cricket-logic checks passed.


## Step 27 — Check that delivery teams belong to that match

A delivery's batting and bowling sides must be the two sides listed for the same match.

In [27]:
match_teams = fact_matches[["match_id", "team1_id", "team2_id"]]
delivery_check = fact_deliveries[["match_id", "batting_team_id", "bowling_team_id"]].merge(
    match_teams, on="match_id", how="left", validate="many_to_one"
)

batting_ok = delivery_check["batting_team_id"].eq(delivery_check["team1_id"]) | delivery_check["batting_team_id"].eq(delivery_check["team2_id"])
bowling_ok = delivery_check["bowling_team_id"].eq(delivery_check["team1_id"]) | delivery_check["bowling_team_id"].eq(delivery_check["team2_id"])

assert batting_ok.all()
assert bowling_ok.all()
assert delivery_check["batting_team_id"].ne(delivery_check["bowling_team_id"]).all()
print("Delivery-to-match team checks passed.")

Delivery-to-match team checks passed.


## Step 28 — Review missing values that are expected

Do not fill unknown optional player attributes or create fake awards for tie/no-result matches.

In [28]:
missing_review = pd.DataFrame({
    "column": ["bowl_style", "field_pos", "player_of_match_id", "player_out_id", "win_by_runs", "win_by_wickets"],
    "missing_rows": [
        dim_players["bowl_style"].isna().sum(),
        dim_players["field_pos"].isna().sum(),
        fact_matches["player_of_match_id"].isna().sum(),
        fact_deliveries["player_out_id"].isna().sum(),
        fact_matches["win_by_runs"].isna().sum(),
        fact_matches["win_by_wickets"].isna().sum(),
    ]
})
display(missing_review)

,column,missing_rows
0,bowl_style,73
1,field_pos,700
2,player_of_match_id,9
3,player_out_id,273886
4,win_by_runs,666
5,win_by_wickets,571


## Step 29 — Export only the six model tables

This folder is dedicated to Power BI model outputs. Old CSVs in this folder are removed so obsolete extra tables cannot remain beside the six current tables.

In [29]:
# Remove old CSV exports from this dedicated output folder.
for old_csv in OUTPUT_DIR.glob("*.csv"):
    old_csv.unlink()

# Write exactly six tables. index=False avoids an extra pandas index column.
for name, table in model_tables.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, date_format="%Y-%m-%d")

print("Export folder:", OUTPUT_DIR.resolve())
print("Files exported:")
for name in model_tables:
    path = OUTPUT_DIR / f"{name}.csv"
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

Export folder: /mnt/data/DATA/POWERBI_TABLES
Files exported:
- dim_teams.csv: 372 bytes
- dim_players.csv: 56,842 bytes
- dim_venues.csv: 2,829 bytes
- dim_date.csv: 364,041 bytes
- fact_matches.csv: 84,200 bytes
- fact_deliveries.csv: 53,331,209 bytes


## Step 30 — Final export check

Reload the exported files and confirm that their shapes match the tables in memory.

In [30]:
for name, table in model_tables.items():
    exported = pd.read_csv(OUTPUT_DIR / f"{name}.csv", low_memory=False)
    assert exported.shape == table.shape, f"Shape changed during export: {name}"

csv_files = sorted(path.name for path in OUTPUT_DIR.glob("*.csv"))
expected_files = sorted(f"{name}.csv" for name in model_tables)
assert csv_files == expected_files, f"Unexpected CSV files in output folder: {csv_files}"

print("Export verified. Exactly six CSV tables are in DATA/POWERBI_TABLES.")

Export verified. Exactly six CSV tables are in DATA/POWERBI_TABLES.


## Power BI relationship guide — using only these six tables

Create **single-direction** relationships:

- `dim_date[date_id]` → `fact_matches[date_id]` (1-to-many)
- `dim_date[date_id]` → `fact_deliveries[date_id]` (1-to-many)
- `dim_venues[venue_id]` → `fact_matches[venue_id]` (1-to-many)
- `dim_venues[venue_id]` → `fact_deliveries[venue_id]` (1-to-many)
- `dim_teams[team_id]` → team-role ID columns in both facts
- `dim_players[player_id]` → player-role ID columns in both facts

Because you asked to keep exactly six tables, the same team/player dimension serves several roles. Set these as the **active** relationships first:

- `dim_teams[team_id]` → `fact_matches[team1_id]`
- `dim_teams[team_id]` → `fact_deliveries[batting_team_id]`
- `dim_players[player_id]` → `fact_matches[player_of_match_id]`
- `dim_players[player_id]` → `fact_deliveries[batter_id]`

Keep other team/player role relationships inactive. Use `USERELATIONSHIP` in a measure when you need a different role, instead of creating extra tables.

### Example — matches won by selected team

```DAX
Matches Won =
CALCULATE(
    COUNTROWS(fact_matches),
    USERELATIONSHIP(dim_teams[team_id], fact_matches[match_winner_id])
)
```

### Example — runs conceded by selected bowling team

```DAX
Runs Conceded =
CALCULATE(
    SUM(fact_deliveries[total_runs]),
    USERELATIONSHIP(dim_teams[team_id], fact_deliveries[bowling_team_id])
)
```

Do **not** create `dim_matches`, `dim_team_aliases`, or extra role-playing tables.

## Final table list

The output folder should contain only:

1. `dim_teams.csv`
2. `dim_players.csv`
3. `dim_venues.csv`
4. `dim_date.csv`
5. `fact_matches.csv`
6. `fact_deliveries.csv`

The original source files remain unchanged.